# Accessing the data
- cleaned demand data
- holiday dictionary

In [ ]:
#Cleaned Demand Data
import sys

sys.path.append("/home/565/pv3484/aus_substation_electricity/data/cleaned_data/VIC")
%cd /home/565/pv3484/aus_substation_electricity/data/cleaned_data/VIC

In [ ]:
#Holiday Dictionary
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity/pia_notebooks")

from VIC_holidays import build_vic_holiday_dict
vh = build_vic_holiday_dict(2004, 2018)

In [ ]:
from VIC_holidays import HOLIDAYS_VIC

def build_holiday_name_map(start_year, end_year):
    out = {}
    for name, fn in HOLIDAYS_VIC.items():
        dates = []
        for year in range(start_year, end_year + 1):
            ts = fn(year)
            if pd.notna(ts):
                dates.append(ts)
        out[name] = sorted(dates)
    return out

holiday_by_name = build_holiday_name_map(2004, 2017)

# Looking at the data first
- to see what I have named the variables
- What I need to call to generate plots

In [ ]:
#Listing all the VIC distributors with cleaned data

import os

vic_root = "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/VIC"

distributors = [
    d for d in os.listdir(vic_root)
    if os.path.isdir(os.path.join(vic_root, d))
]

distributors

In [ ]:
#Looking inside one (Jemena) cleaned demand

dist = "Jemena_VIC"
dist_path = os.path.join(vic_root, dist)

csvs = [f for f in os.listdir(dist_path) if f.endswith("_cleaned.csv")]
csvs

In [ ]:
#Reading one csv file to see column names

import pandas as pd

csv_path = "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/VIC/Jemena_VIC/Jemena_VIC_2012_cleaned.csv"

df = pd.read_csv(csv_path)
df.columns

In [ ]:
#Converts 'StartDeliveryTime' column into real datetime (converting them into proper pandas datetime object)

import pandas as pd

def load_vic_csv(path):
    df = pd.read_csv(path)

    # Parse the VIC timestamp column
    df['timestamp'] = pd.to_datetime(df['StartDeliveryTime'])

    # Set as index for easy time‑series work (converts Dataframe into time-series dataframe to allow for time-series analysis)
    df = df.set_index('timestamp')

    return df

In [ ]:
path = "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/VIC/Jemena_VIC/Jemena_VIC_2012_cleaned.csv"
df = load_vic_csv(path)
df.head()

In [ ]:
df.columns #view columns
df.head

# Creating demand plots for all public holidays in VIC
- Testing with Powercor_VIC first
- Using cleaned VIC demand
- demand for each holiday function

## Loading in metadata to access full substation name

In [ ]:
import pandas as pd

meta_path = "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/VIC/Powercor_VIC/Powercor_VIC_metadata.csv"

meta = pd.read_csv(meta_path)

In [ ]:
meta.head()

In [ ]:
#Load and map full substation names
substation_name_map = dict(zip(meta["ID"], meta["Zone Substation Name"]))

In [ ]:
substation_name_map["ART"]

## Plotting Demans vs time for all distributors and substations

In [ ]:
# Load the distributors
from pathlib import Path
import pandas as pd

def load_distributor(dist_path: Path) -> pd.DataFrame:
    frames = []
    for csv_file in dist_path.glob("*_cleaned.csv"):
        df = pd.read_csv(csv_file)
        df["StartDeliveryTime"] = pd.to_datetime(df["StartDeliveryTime"])
        df["year"] = df["StartDeliveryTime"].dt.year
        df["source_file"] = csv_file.name
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

In [ ]:
#Loading metadata for the full substation names
def load_metadata(dist_path: Path):
    meta_file = list(dist_path.glob("*_metadata.csv"))[0]
    meta = pd.read_csv(meta_file)
    return dict(zip(meta["ID"], meta["Zone Substation Name"]))

In [ ]:
#Plotting holiday function
import matplotlib.pyplot as plt

def plot_holiday_one_sub(df, substation, holiday_dict, full_name):
    hours = list(range(24))

    figs = {}  # holiday_name → figure

    for holiday_name, date_list in holiday_dict.items():
        fig, ax = plt.subplots(figsize=(12, 5))

        for ts in date_list:
            year = ts.year
            mask = df["StartDeliveryTime"].dt.date == ts.date()
            day_df = df.loc[mask, ["StartDeliveryTime", substation]]

            if day_df.empty:
                continue

            day_df["hour"] = day_df["StartDeliveryTime"].dt.hour

            hourly = (
                day_df.groupby("hour")[substation]
                      .mean()
                      .reindex(hours)
            )

            ax.plot(hours, hourly, label=str(year), linewidth=1.2)

        ax.set_title(f"{full_name} — {holiday_name} (Hourly Demand Across Years)")
        ax.set_xlabel("Hour of Day")
        ax.set_ylabel("Demand (MW)")
        ax.set_xticks(hours)
        ax.legend(title="Year")
        fig.tight_layout()

        figs[holiday_name] = fig

    return figs

In [ ]:
#Loop through each distributor and save as multi-page pdfs
from matplotlib.backends.backend_pdf import PdfPages

VIC_ROOT = Path("/home/565/pv3484/aus_substation_electricity/data/cleaned_data/VIC")
OUT_ROOT = Path("/home/565/pv3484/aus_substation_electricity/figures/demand_pubhols/VIC")

for dist_dir in VIC_ROOT.iterdir():
    if not dist_dir.is_dir():
        continue
    if dist_dir.name.startswith(".") or dist_dir.name == "weather_obs":
        continue

    distributor = dist_dir.name
    print(f"Processing {distributor}")

    # Load data + metadata
    df = load_distributor(dist_dir)
    name_map = load_metadata(dist_dir)

    # Identify substation columns
    sub_cols = [
        c for c in df.columns
        if c not in ["StartDeliveryTime", "year", "source_file", "distributor"]
    ]

    # Create output folder
    out_dir = OUT_ROOT / distributor
    out_dir.mkdir(parents=True, exist_ok=True)

    # For each holiday, create a multi-page PDF
    for holiday_name in holiday_by_name.keys():
        pdf_path = out_dir / f"{holiday_name.replace(' ', '_')}.pdf"

        with PdfPages(pdf_path) as pdf:
            for sub in sub_cols:
                full_name = name_map.get(sub, sub)
                figs = plot_holiday_one_sub(df, sub, {holiday_name: holiday_by_name[holiday_name]}, full_name)
                pdf.savefig(figs[holiday_name])
                plt.close(figs[holiday_name])